# 🧩 Notebook 1 · Tokenización: de texto a números

**Diplomado CENIA · ¿Cómo lee una red neuronal?** · Acto 1 de la clase

Una red neuronal solo entiende números. En este notebook recorremos el camino de la clase:

**texto → tokens → IDs → vectores**

1. Cortar por **palabras** y ver aparecer el `[UNK]`.
2. Cortar por **caracteres**.
3. Programar **BPE desde cero**.
4. Mirar cómo cortan el texto los **tokenizadores reales** (GPT, BERT, XLM-R).
5. **Entrenar nuestro propio tokenizador** con el Quijote.
6. Pasar de **IDs a vectores** con una capa de embedding.

In [ ]:
import re, time, math, random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0); random.seed(0)
print("Usando:", device)

## 1. Idea 1: cortar por palabras

Lo más simple: separar por espacios (y signos de puntuación) y darle un número a cada palabra distinta. Usamos un corpus mini de juguete.

In [ ]:
corpus_mini = """En la mañana tomo café. Me gusta el café con leche y sin azúcar. A mi hermana no le gusta el café,
ella toma té con azúcar. Mi abuela tomaba café con mucha azúcar. Los cafés de la esquina son buenos.
Tengo un gato y una gata. Los gatos duermen todo el día. Ayer llegó un gatito nuevo y los gatitos juegan mucho.
El perro del vecino persigue a los gatos, pero los perros también duermen. El perrito juega en el parque.
Los niños juegan en el parque. Yo jugaba en el parque cuando era niño. Ahora trabajo en la ciudad.
Mis amigos trabajan en el centro. Nosotros comemos juntos los viernes. Ellos comen pescado y nosotros comemos pan."""

def separar_palabras(texto):
    # palabras y signos de puntuación por separado, todo en minúsculas
    return re.findall(r"\w+|[^\w\s]", texto.lower())

palabras = separar_palabras(corpus_mini)
vocab_palabras = {"[UNK]": 0}
for p in palabras:
    if p not in vocab_palabras:
        vocab_palabras[p] = len(vocab_palabras)

print(f"{len(palabras)} palabras en el corpus, {len(vocab_palabras)} distintas")
print(list(vocab_palabras.items())[:12], "...")

In [ ]:
def codificar_palabras(texto):
    return [vocab_palabras.get(p, 0) for p in separar_palabras(texto)]

for frase in ["No me gusta el café sin azúcar", "Ayer googleé cómo chatGPTear, cachai"]:
    ids = codificar_palabras(frase)
    tokens = [p if i != 0 else "[UNK]" for p, i in zip(separar_palabras(frase), ids)]
    print(frase, "\n  →", tokens, "\n  →", ids, "\n")

Cualquier palabra que no estaba en el corpus se convierte en `[UNK]` y el modelo pierde esa información por completo. Además, "gato", "gatos" y "gatito" son IDs sin ninguna relación entre sí.

## 2. Idea 2: cortar por caracteres

In [ ]:
frase = "No me gusta el café sin azúcar"
print(list(frase))
print(f"{len(separar_palabras(frase))} palabras → {len(frase)} tokens de caracteres")

Nunca hay `[UNK]` (con ~100 símbolos se escribe cualquier cosa), pero las secuencias quedan **4–5 veces más largas** y una letra sola casi no significa nada.

---

## 3. Idea 3: BPE desde cero

Implementemos el algoritmo de la clase (basado en el [curso de 🤗 Hugging Face, cap. 6.5](https://huggingface.co/learn/llm-course/chapter6/5)):

1. Contar las palabras del corpus y partir cada una en caracteres.
2. Contar **pares de vecinos**.
3. **Juntar** el par más frecuente en un token nuevo. Repetir.

Empezamos con el corpus de la diapositiva "BPE paso a paso", para poder comparar.

In [ ]:
# Paso 1: palabras con su frecuencia, y cada palabra partida en caracteres
frecuencias = {"gato": 6, "gatos": 4, "gata": 3, "gatito": 2, "pato": 5, "patos": 2}
partes = {palabra: list(palabra) for palabra in frecuencias}
partes

In [ ]:
# Paso 2: contar pares de vecinos (cada par cuenta tantas veces como aparece su palabra)
def contar_pares(partes, frecuencias):
    pares = Counter()
    for palabra, freq in frecuencias.items():
        simbolos = partes[palabra]
        for a, b in zip(simbolos, simbolos[1:]):
            pares[(a, b)] += freq
    return pares

contar_pares(partes, frecuencias).most_common(5)

El par más frecuente es `a + t` (22 veces): ese será el primer token nuevo, igual que en la diapositiva.

**Paso 3:** `juntar_par` recorre los símbolos de cada palabra y, cada vez que encuentra `a` seguido de `b`, los reemplaza por un solo símbolo `a + b`.

In [ ]:
def juntar_par(a, b, partes):
    nuevas = {}
    for palabra, simbolos in partes.items():
        resultado = []
        i = 0
        while i < len(simbolos):
            if i < len(simbolos) - 1 and simbolos[i] == a and simbolos[i + 1] == b:
                resultado.append(a + b)
                i += 2
            else:
                resultado.append(simbolos[i])
                i += 1
        nuevas[palabra] = resultado
    return nuevas

juntar_par("a", "t", partes)

Con eso, entrenar BPE es solo un loop. Las reglas son las mismas de la diapositiva:

In [ ]:
def entrenar_bpe(frecuencias, n_merges, verbose=True):
    partes = {palabra: list(palabra) for palabra in frecuencias}
    reglas = []
    for paso in range(n_merges):
        pares = contar_pares(partes, frecuencias)
        if not pares:
            break
        (a, b), veces = pares.most_common(1)[0]
        partes = juntar_par(a, b, partes)
        reglas.append((a, b))
        if verbose:
            print(f"{paso + 1:2d}. {a} + {b} → {a + b:8s} ({veces} veces)")
    return reglas, partes

reglas, partes_finales = entrenar_bpe(frecuencias, 6)
partes_finales

Para tokenizar una palabra **nueva**, aplicamos las reglas **en el mismo orden** en que se aprendieron:

In [ ]:
def tokenizar_bpe(palabra, reglas):
    simbolos = list(palabra)
    for a, b in reglas:
        simbolos = juntar_par(a, b, {"_": simbolos})["_"]
    return simbolos

for palabra in ["gatas", "patito", "patitos", "zapato"]:
    print(f"{palabra:8s} →", tokenizar_bpe(palabra, reglas))

Ningún `[UNK]`: las palabras nuevas se arman con piezas conocidas (en el peor caso, con letras sueltas).

Ahora con el corpus mini y 60 reglas. ¿Qué piezas aprende?

In [ ]:
frecuencias_mini = Counter(p for p in separar_palabras(corpus_mini) if p.isalpha())
reglas_mini, _ = entrenar_bpe(frecuencias_mini, 60, verbose=False)
print("Primeras 20 reglas:", ["".join(r) for r in reglas_mini[:20]], "\n")
for palabra in ["gatito", "comeremos", "chatgptear", "cachai"]:
    print(f"{palabra:12s} →", tokenizar_bpe(palabra, reglas_mini))

💡 **¿BPE "sabe" que *-ito* es un diminutivo o que *-mos* es primera persona plural?**

No. BPE solo **cuenta** pares frecuentes. Si aparece una pieza que coincide con un sufijo es porque ese sufijo es frecuente, no porque el algoritmo entienda gramática. Además, el tokenizador se entrena **antes** y **aparte** del modelo: no usa backprop.


---

## 4. Tokenizadores reales

### 4.1 Así ve el texto GPT-4o

`tiktoken` es la librería de OpenAI. `o200k_base` es el tokenizador de GPT-4o (≈200 mil tokens). Esta función pinta cada token de un color, como [tiktokenizer.vercel.app](https://tiktokenizer.vercel.app):

In [ ]:
import tiktoken
from IPython.display import HTML, display

gpt4o = tiktoken.get_encoding("o200k_base")
COLORES = ["#ffd6a5", "#caffbf", "#9bf6ff", "#bdb2ff", "#ffc6ff", "#fdffb6"]

def mostrar_tokens(texto, enc=gpt4o):
    ids = enc.encode(texto)
    piezas = [enc.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in ids]
    spans = "".join(
        f'<span title="ID {i}" style="background:{COLORES[k % 6]};padding:2px 1px;border-radius:3px;'
        f'font-family:monospace;font-size:15px;white-space:pre;color:#1d2433">{p.replace(" ", "␣")}</span>'
        for k, (i, p) in enumerate(zip(ids, piezas)))
    display(HTML(f'<div style="margin:4px 0">{spans} <small>· {len(ids)} tokens</small></div>'))
    return ids

for texto in ["How many r's are in strawberry?", "ferrocarrilero", "No me gusta el café sin azúcar",
              "1234567 + 2024", "hola Hola HOLA", "¿Cómo estás? 😊"]:
    mostrar_tokens(texto)

In [ ]:
print("strawberry con espacio adelante:", gpt4o.encode(" strawberry"), "← ¡un solo número!")
print("ferrocarrilero:", [gpt4o.decode([i]) for i in gpt4o.encode("ferrocarrilero")], "← las 5 r están escondidas dentro de 4 tokens")

### El "impuesto" del español

¿Cuántos tokens cuesta la misma frase en español y en inglés? Comparamos el tokenizador de GPT-4 (`cl100k_base`) con el de GPT-4o (`o200k_base`). Se pueden agregar a la lista frases que propongan en el chat.

In [ ]:
pares_es_en = [
    ("No me gusta el café sin azúcar.", "I don't like coffee without sugar."),
    ("Los perros que vi ayer en el parque estaban cansados.", "The dogs I saw yesterday in the park were tired."),
    ("¿Dónde está la estación de trenes más cercana?", "Where is the nearest train station?"),
    # ("frase del chat", "its English translation"),
]

for nombre in ["cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(nombre)
    total_es = sum(len(enc.encode(es)) for es, en in pares_es_en)
    total_en = sum(len(enc.encode(en)) for es, en in pares_es_en)
    print(f"{nombre}: español {total_es} tokens · inglés {total_en} tokens → {total_es / total_en:.2f}×")

Más tokens = más caro (las APIs cobran por token), más lento y cabe menos texto en el contexto. El tokenizador más nuevo reduce la brecha, pero no la elimina.

### 4.2 Las tres familias: BPE, WordPiece y SentencePiece

Cada familia marca los límites de palabra de forma distinta:
- **GPT-2** (BPE por bytes): `Ġ` = "aquí había un espacio".
- **BETO**, el BERT en español de la U. de Chile (WordPiece): `##` = "esto continúa la palabra anterior".
- **XLM-RoBERTa** (SentencePiece): `▁` = "aquí empieza una palabra".

(La primera vez descarga los tokenizadores desde 🤗 Hugging Face, unos segundos.)

In [ ]:
from transformers import AutoTokenizer

tokenizadores = {
    "GPT-2 (BPE)": AutoTokenizer.from_pretrained("openai-community/gpt2"),
    "BETO (WordPiece)": AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased"),
    "XLM-R (SentencePiece)": AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base"),
}

frase = "Los otorrinolaringólogos ¿comían ñandú?"
for nombre, tok in tokenizadores.items():
    piezas = tok.tokenize(frase)
    print(f"{nombre:22s} {len(piezas):2d} tokens: {piezas}")

💡 **¿Qué es ese `Ã³` en GPT-2?** GPT-2 trabaja con **bytes**, no con letras. La "ó" ocupa 2 bytes en UTF-8 y GPT-2 (entrenado casi solo en inglés) nunca aprendió a juntarlos, así que la muestra como dos símbolos raros. Igual nunca pierde información:

In [ ]:
gpt2 = tokenizadores["GPT-2 (BPE)"]
ids = gpt2.encode(frase)
print(ids)
print(gpt2.decode(ids))   # vuelve exacto al texto original

---

## 5. Entrenar nuestro propio tokenizador

GPT-2 se entrenó con texto en inglés, por eso corta mal el español. ¿Qué pasa si **re-entrenamos** el mismo algoritmo con texto en español? Usemos el Quijote completo (≈2 millones de caracteres):

In [ ]:
import urllib.request

def descargar_quijote():
    urls = ["https://www.gutenberg.org/cache/epub/2000/pg2000.txt",
            "https://gutenberg.pglaf.org/2/0/0/2000/2000-0.txt"]
    for url in urls:
        try:
            crudo = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")
            inicio = crudo.index("\n", crudo.index("*** START")) + 1
            return crudo[inicio:crudo.index("*** END")].strip().replace("\r\n", "\n")
        except Exception as e:
            print("No se pudo con", url, "→", e)
    raise RuntimeError("No se pudo descargar el Quijote")

quijote = descargar_quijote()
print(f"{len(quijote):,} caracteres")
print(quijote[2000:2400])

In [ ]:
def lotes_de_texto(texto, tam=10_000):
    for i in range(0, len(texto), tam):
        yield texto[i:i + tam]

tok_quijote = gpt2.train_new_from_iterator(lotes_de_texto(quijote), vocab_size=8000)

frase = "En un lugar de la Mancha, de cuyo nombre no quiero acordarme, vivía un hidalgo"
print("GPT-2 original :", len(gpt2.tokenize(frase)), gpt2.tokenize(frase), "\n")
print("GPT-2 Quijote  :", len(tok_quijote.tokenize(frase)), tok_quijote.tokenize(frase))

Mismo algoritmo, otros datos → otro vocabulario. Ahora "Mancha" e "hidalgo" son un solo token. **El tokenizador refleja el texto con el que se entrenó.**

### ⭐ Extra: el espectro carácter ↔ palabra

Entrenamos tokenizadores con distintos tamaños de vocabulario y miramos cuántos tokens necesita la misma frase. Es el slider de la clase, ahora con datos reales:

In [ ]:
tamanos = [300, 1000, 4000, 16000, 32000]
n_tokens = []
for v in tamanos:
    t = gpt2.train_new_from_iterator(lotes_de_texto(quijote), vocab_size=v)
    n_tokens.append(len(t.tokenize(frase)))
    print(f"vocabulario {v:6,d} → {n_tokens[-1]:3d} tokens: {t.tokenize(frase)[:12]}...")

plt.figure(figsize=(6, 3.5))
plt.semilogx(tamanos, n_tokens, "o-")
plt.xlabel("tamaño del vocabulario"); plt.ylabel("tokens para la frase")
plt.title("Más vocabulario → secuencias más cortas"); plt.grid(alpha=.3); plt.show()

---

## 6. De IDs a vectores

El último paso: cada ID se convierte en un vector buscando su fila en una **tabla de embeddings** (`nn.Embedding`). Esos números se aprenden con backprop, como cualquier peso.

In [ ]:
ids = torch.tensor(tok_quijote.encode("No me gusta el café sin azúcar"))
print("IDs:", ids.tolist())

tabla = nn.Embedding(num_embeddings=len(tok_quijote), embedding_dim=8)   # 8 números por token (en GPT-4o son miles)
vectores = tabla(ids)
print("forma:", tuple(vectores.shape), "→ (tokens, dimensión)")
print(vectores[:2])

💡 **¿Es algo nuevo?** No: buscar la fila es lo mismo que multiplicar un vector **one-hot** por una matriz, o sea, una **capa densa** sin sesgo. Comprobémoslo:

In [ ]:
one_hot = F.one_hot(ids, num_classes=len(tok_quijote)).float()   # (tokens, vocabulario): puros 0 y un 1
como_capa_densa = one_hot @ tabla.weight                           # (tokens, 8)
print("¿Iguales?", torch.allclose(como_capa_densa, vectores))

`nn.Embedding` hace exactamente eso, pero sin construir el one-hot gigante: solo busca la fila.

---

## ✅ Resumen

| Paso | Lo que vimos |
|---|---|
| texto → tokens | palabras (`[UNK]`), caracteres (secuencias largas) y **BPE** (piezas frecuentes) |
| tokens → IDs | `tiktoken`, GPT-2, BETO, XLM-R, y un tokenizador entrenado con el Quijote |
| IDs → vectores | `nn.Embedding` = buscar una fila = one-hot × matriz |

➡️ En el **notebook 2** le damos esa lista de vectores a una red con **memoria**.